In [1]:
import pandas as pd
import sqlite3

In [4]:
import pandas as pd

df = pd.read_csv('inventory_data_for_database.csv', sep='\t')
df.to_csv('fixed.csv', sep=',', index=False)

In [5]:
import csv

with open('fixed.csv', 'r') as f_in, open('data_for_mysql.csv', 'w', newline='') as f_out:
    reader = csv.reader(f_in, quotechar='"')
    writer = csv.writer(f_out)

    # Skip header if needed
    next(reader)

    for row in reader:
        # Remove outer quotes from each field
        cleaned = [field.strip('"') for field in row]
        writer.writerow(cleaned)

In [2]:
# 1. Load CSV file
csv_path = "./before_preprocessed_inventory_data.csv"  # Replace with your actual CSV file path
df = pd.read_csv(csv_path)

# 2. Convert Date to ISO format (YYYY-MM-DD)
df['Date'] = pd.to_datetime(df['Date'], format='%d/%m/%Y').dt.strftime('%Y-%m-%d')

# 3. Connect to (or create) the new SQLite database
conn = sqlite3.connect('inventory.db')
cursor = conn.cursor()

# 4. Create the inventory table
cursor.execute("""
               CREATE TABLE IF NOT EXISTS inventory
               (
                   Date
                   TEXT,
                   Store_ID
                   TEXT,
                   Product_ID
                   TEXT,
                   Category
                   TEXT,
                   Region
                   TEXT,
                   Inventory_Level
                   INTEGER,
                   Units_Sold
                   INTEGER,
                   Units_Ordered
                   INTEGER,
                   Demand_Forecast
                   REAL,
                   Price
                   REAL,
                   Discount
                   REAL,
                   Weather_Condition
                   TEXT,
                   Holiday_Promotion
                   INTEGER,
                   Competitor_Pricing
                   REAL,
                   Seasonality
                   TEXT,
                   Lead_Time
                   INTEGER
               );
               """)

# 5. Insert the data
df.to_sql('inventory', conn, if_exists='append', index=False)

# 6. Optional: verify inserted data
print(pd.read_sql_query("SELECT * FROM inventory LIMIT 5", conn))

# 7. Close the connection
conn.close()

         Date Store_ID Product_ID     Category Region  Inventory_Level  \
0  2022-01-01     S001      P0001    Groceries  North              231   
1  2022-01-01     S001      P0002         Toys  South              204   
2  2022-01-01     S001      P0003         Toys   West              102   
3  2022-01-01     S001      P0004         Toys  North              469   
4  2022-01-01     S001      P0005  Electronics   East              166   

   Units_Sold  Units_Ordered  Demand_Forecast  Price  Discount  \
0         127             55           135.47  33.50      20.0   
1         150             66           144.04  63.01      20.0   
2          65             51            74.02  27.99      10.0   
3          61            164            62.18  32.72      10.0   
4          14            135             9.26  73.64       0.0   

  Weather_Condition  Holiday_Promotion  Competitor_Pricing Seasonality  \
0             Rainy                  0               29.69      Autumn   
1         

In [3]:
conn = sqlite3.connect("inventory.db")
cursor = conn.cursor()

# Execute the update
cursor.execute("""
               UPDATE inventory
               SET Inventory_Level = ?
               WHERE Store_ID = ?
                 AND Product_ID = ?
                 AND Date = ?;
               """, (55, 'S001', 'P0004', '2022-02-28'))

conn.commit()
conn.close()

In [4]:
conn = sqlite3.connect("inventory.db")
cursor = conn.cursor()

# Execute the SELECT query
cursor.execute("""
               SELECT Inventory_Level
               FROM inventory
               WHERE Store_ID = ?
                 AND Product_ID = ?
                 AND Date = ?;
               """, ('S001', 'P0004', '2022-02-28'))

# Fetch the result
result = cursor.fetchone()  # or use fetchall() for multiple rows

# Handle and print the result
if result:
    print("Inventory Level:", result[0])
else:
    print("No data found.")

# Close the connection
conn.close()

Inventory Level: 55
